# Comparing Matches with Ground Truth

We were given a csv of groud truth matches, which has pairs of 1870 and 1880 matches with a expert decided confidence level of how good the match is. We want to compare these true matches with our predicted matches. In this notebook, we explored the success of matches when setting predicted matches are comparitsions with a match probablity greater than 0.80 and then match probabilty greater than 0.90. For each match probabilty threshold, we explored what percentage of true matches overall and them for each confidence level. Results show that our model is somewhhat successful at matching records and has different successful levels across ground truth confidecne levels.

## Setup and Imports

In [188]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

In [189]:
predictions = pd.read_parquet(
    "../data/boosted_preds.parquet",
    engine="fastparquet"
)


In [190]:
predictions.head()

,unique_id_l,unique_id_r,match_probability,relatives_match_probability
0,ALB-CN-1870-10570,ALB-CN-1880-6166,0.999702,NaN
1,ALB-CN-1870-12185,ALB-CN-1880-6171,0.999702,NaN
2,ALB-CN-1870-14277,ALB-CN-1880-6173,0.999702,NaN
3,ALB-CN-1870-18074,ALB-CN-1880-6174,0.999702,NaN
4,ALB-CN-1870-14278,ALB-CN-1880-6175,0.999702,NaN


In [191]:
truth = pd.read_csv('../data/ground_truth.csv')
truth['1870_id']= 'ALB-CN-1870-' + truth['1870_line'].astype(str)
truth['1880_id']= 'ALB-CN-1880-' + truth['1880_line'].astype(str)

In [192]:
truth.head()

,1870_line,1880_line,score,confidence,1870_id,1880_id
0,1688,22721,495,3,ALB-CN-1870-1688,ALB-CN-1880-22721
1,1695,22737,480,3,ALB-CN-1870-1695,ALB-CN-1880-22737
2,1693,22735,470,3,ALB-CN-1870-1693,ALB-CN-1880-22735
3,1692,22734,460,3,ALB-CN-1870-1692,ALB-CN-1880-22734
4,17144,25766,455,3,ALB-CN-1870-17144,ALB-CN-1880-25766


In [193]:
mentions = pd.read_csv('../data/mentions.csv')
mentions.head()

/var/folders/p9/9vqlyg213v3b9h9zrksgz6z00000gn/T/ipykernel_5746/2722962375.py:1: DtypeWarning:

Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.



,mention_id,source,source_year,county,original_data,confidence,full_name,first_name,middle_name,last_name,...,norm_occupation,enslaver_id,location_id,head,household_id,family_id,created,narrative,soundex_last_name,narrative_vector
0,ALB-CH-1851-2071,ALB_CH_1851,1851,ALB,"{""line"": ""2071"", ""race"": ""B"", ""gender"": ""F"", ""...",0.80,Martha,Martha,NaN,NaN,...,NaN,NaN,NaN,f,NaN,NaN,2026-06-20 17:49:32.846283+00,Martha (F / B in Alb). Enslaved by: Mary Moore.,NaN,NaN
1,ALB-VR-1715-4362,ALB_VR_1715,1868,ALB,"{""line"": ""4362"", ""note"": """", ""race"": ""B"", ""typ...",0.84,Nicey Ann Coles,Nicey,Ann,Coles,...,NaN,NaN,NaN,f,NaN,NaN,2026-06-20 17:34:55.49837+00,NaN,C420,NaN
2,ALB-VR-1715-4362.1,ALB_VR_1715,1868,ALB,"{""line"": ""4362"", ""note"": """", ""race"": ""B"", ""typ...",0.85,Coles,Coles,NaN,Coles,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-20 17:36:09.428451+00,NaN,C420,NaN
3,ALB-VR-1715-4362.2,ALB_VR_1715,1868,ALB,"{""line"": ""4362"", ""note"": """", ""race"": ""B"", ""typ...",0.85,Coles,Coles,NaN,Coles,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-20 17:36:45.128086+00,NaN,C420,NaN
4,ALB-VR-1715-4362.3,ALB_VR_1715,1868,ALB,"{""line"": ""4362"", ""note"": """", ""race"": ""B"", ""typ...",0.85,Coles,Coles,NaN,Coles,...,NaN,NaN,NaN,NaN,NaN,NaN,2026-06-20 17:36:51.894296+00,NaN,C420,NaN


First, let's explore what the different confidnece levels look like.

## Look at an example of a confidence with 3

In [194]:
mentions[mentions['mention_id'] == 'ALB-CN-1870-1688' ].T

,55503
mention_id,ALB-CN-1870-1688
source,ALB_CN_1870
source_year,1870
county,ALB
original_data,"{""age"": ""38"", ""head"": ""Y"", ""line"": ""1688"", ""pa..."
confidence,0.9
full_name,Dabney Johnson
first_name,Dabney
middle_name,NaN
last_name,Johnson


In [195]:
mentions[mentions['mention_id'] == 'ALB-CN-1880-22721' ].T

,78316
mention_id,ALB-CN-1880-22721
source,ALB_CN_1880
source_year,1880
county,ALB
original_data,"{""age"": ""48"", ""head"": ""Y"", ""line"": ""22721"", ""r..."
confidence,0.9
full_name,Dabney Johnson
first_name,Dabney
middle_name,NaN
last_name,Johnson


This seems to be a very good match. The records have the same name. Birth year is exact and occupations match. This is a specfic match we have to further verify with relatives however because there were multiple Dabney Johnson matches in our predicted matches.

## Look at an example of confidence 2

In [196]:
truth[truth['confidence'] == 2].head(3)

,1870_line,1880_line,score,confidence,1870_id,1880_id
25,17149,25770,420,2,ALB-CN-1870-17149,ALB-CN-1880-25770
48,18766,10837,405,2,ALB-CN-1870-18766,ALB-CN-1880-10837
63,538,16327,100,2,ALB-CN-1870-538,ALB-CN-1880-16327


In [197]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-17149'].T


,55799
mention_id,ALB-CN-1870-17149
source,ALB_CN_1870
source_year,1870
county,ALB
original_data,"{""age"": ""1"", ""head"": """", ""line"": ""17149"", ""pag..."
confidence,0.9
full_name,William Sammons
first_name,William
middle_name,NaN
last_name,Sammons


In [198]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-25770'].T

,85939
mention_id,ALB-CN-1880-25770
source,ALB_CN_1880
source_year,1880
county,ALB
original_data,"{""age"": ""9"", ""head"": """", ""line"": ""25770"", ""rac..."
confidence,0.9
full_name,William Sammons
first_name,William
middle_name,NaN
last_name,Sammons


In [199]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-17149']['birth_year']


55799    1869.0
Name: birth_year, dtype: float64

In [200]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-25770']['birth_year']

85939    1871.0
Name: birth_year, dtype: float64

Although the first and last name are the same, occupation is missing in 1870, so we cannot confirm it is the same. Birth year is 2 years apart so this is a probable match, but not as good as the one we looked at with confidence 3.

## Look at an Example of Confidence 1

In [201]:
truth.loc[truth['confidence'] == 1].head(3)

,1870_line,1880_line,score,confidence,1870_id,1880_id
24,17478,22724,420,1,ALB-CN-1870-17478,ALB-CN-1880-22724
52,4121,23484,100,1,ALB-CN-1870-4121,ALB-CN-1880-23484
55,12513,24928,100,1,ALB-CN-1870-12513,ALB-CN-1880-24928


In [202]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-17478'].T

,56167
mention_id,ALB-CN-1870-17478
source,ALB_CN_1870
source_year,1870
county,ALB
original_data,"{""age"": ""5"", ""head"": """", ""line"": ""17478"", ""pag..."
confidence,0.9
full_name,George Johnson
first_name,George
middle_name,NaN
last_name,Johnson


In [203]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-22724'].T

,52692
mention_id,ALB-CN-1880-22724
source,ALB_CN_1880
source_year,1880
county,ALB
original_data,"{""age"": ""15"", ""head"": """", ""line"": ""22724"", ""ra..."
confidence,0.9
full_name,George Johnson
first_name,George
middle_name,NaN
last_name,Johnson


## Look at Confidence at 0

In [204]:
truth.loc[truth['confidence'] == 0].head(3)

,1870_line,1880_line,score,confidence,1870_id,1880_id
53,16276,20673,100,0,ALB-CN-1870-16276,ALB-CN-1880-20673
54,10293,31262,100,0,ALB-CN-1870-10293,ALB-CN-1880-31262
58,21086,5367,100,0,ALB-CN-1870-21086,ALB-CN-1880-5367


In [205]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-16276'].T

,81363
mention_id,ALB-CN-1870-16276
source,ALB_CN_1870
source_year,1870
county,ALB
original_data,"{""age"": ""12"", ""head"": """", ""line"": ""16276"", ""pa..."
confidence,0.9
full_name,James Washington
first_name,James
middle_name,NaN
last_name,Washington


In [206]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-20673'].T

,96572
mention_id,ALB-CN-1880-20673
source,ALB_CN_1880
source_year,1880
county,ALB
original_data,"{""age"": ""7"", ""head"": """", ""line"": ""20673"", ""rac..."
confidence,0.9
full_name,James Washington
first_name,James
middle_name,NaN
last_name,Washington


It makes sense that this match confidence is 0. Although, first name, last name, race, and gender is the same, year is way off and we can't verify that occupation is the same

## Comparing predicted matches to confirmed matches

### Threshold at 0.8

In [207]:
threshold =  0.80
threshold

0.8

In [208]:
# Making pairs of all predicted matches (1870_unique_id, 1880_unique_id)
matches = predictions[predictions['match_probability'] > threshold]
matches_pairs = list(zip(matches['unique_id_l'], matches['unique_id_r']))
matches_pairs[:3]



[('ALB-CN-1870-10570', 'ALB-CN-1880-6166'),
 ('ALB-CN-1870-12185', 'ALB-CN-1880-6171'),
 ('ALB-CN-1870-14277', 'ALB-CN-1880-6173')]

In [209]:
# Making pairs of true matches (1870_unique_id, 1880_unique_id)
truth_pairs = list(zip(truth['1870_id'], truth['1880_id']))
truth_pairs[:3]

[('ALB-CN-1870-1688', 'ALB-CN-1880-22721'),
 ('ALB-CN-1870-1695', 'ALB-CN-1880-22737'),
 ('ALB-CN-1870-1693', 'ALB-CN-1880-22735')]

How much of the true pairs did we correctly match?

In [210]:
# set(matches_pairs & set(truth_pairs)) takes each unique values that are common in both sets and we divide by the length of truth pairs for a proprtion
proportion_matched = len(set(matches_pairs) & set(truth_pairs)) / len(truth_pairs)
proportion_matched

0.8385269121813032

#### Looking at metrics for specfic confidence levels

##### Confidence Level 3

In [211]:
# Confidence 3 true match (1870_unique_id, 1880_unique_id)
truth_confidence_3 = truth[truth['confidence'] == 3]
truth_pairs_3 = list(zip(truth_confidence_3['1870_id'], truth_confidence_3['1880_id']))
truth_pairs_3[:3]


[('ALB-CN-1870-1688', 'ALB-CN-1880-22721'),
 ('ALB-CN-1870-1695', 'ALB-CN-1880-22737'),
 ('ALB-CN-1870-1693', 'ALB-CN-1880-22735')]

In [212]:
matches_pairs[:3]

[('ALB-CN-1870-10570', 'ALB-CN-1880-6166'),
 ('ALB-CN-1870-12185', 'ALB-CN-1880-6171'),
 ('ALB-CN-1870-14277', 'ALB-CN-1880-6173')]

In [213]:
proportion_matched_3 = len(set(matches_pairs) & set(truth_pairs_3)) / len(truth_pairs_3)
proportion_matched_3

0.9951456310679612

In [214]:
set(truth_pairs_3) - set(matches_pairs)

{('ALB-CN-1870-8855', 'ALB-CN-1880-17581')}

In [215]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1870-8855'].T


,67901
mention_id,ALB-CN-1870-8855
source,ALB_CN_1870
source_year,1870
county,ALB
original_data,"{""age"": ""5"", ""head"": """", ""line"": ""8855"", ""page..."
confidence,0.9
full_name,Susan A Timberlake
first_name,Susan
middle_name,A
last_name,Timberlake


In [216]:
mentions.loc[mentions['mention_id'] == 'ALB-CN-1880-17581'].T

,36986
mention_id,ALB-CN-1880-17581
source,ALB_CN_1880
source_year,1880
county,ALB
original_data,"{""age"": ""18"", ""head"": """", ""line"": ""17581"", ""ra..."
confidence,0.9
full_name,A Susan Timberlake
first_name,A
middle_name,Susan
last_name,Timberlake


This was the one match with confidence 3, we failed to match. Looking at the actual mentions record it makes sense. In the 1880 census mention, the first and middle name are flipped, so when we compare norm first name in our model we failed to catch it. This is a good point that our models will not correctly match all people due to some encoding errors.

##### Confidence Level 2

In [217]:
truth_confidence_2 = truth[truth['confidence'] == 2]
truth_pairs_2 = list(zip(truth_confidence_2['1870_id'], truth_confidence_2['1880_id']))
truth_pairs_2[:3]

[('ALB-CN-1870-17149', 'ALB-CN-1880-25770'),
 ('ALB-CN-1870-18766', 'ALB-CN-1880-10837'),
 ('ALB-CN-1870-538', 'ALB-CN-1880-16327')]

In [218]:
proportion_matched_2 = len(set(matches_pairs) & set(truth_pairs_2)) / len(truth_pairs_2)
proportion_matched_2

1.0

We successfully matched all people who are matched at confidence 2.

##### Confidence Level 1

In [219]:
truth_confidence_1 = truth[truth['confidence'] == 1]
truth_pairs_1 = list(zip(truth_confidence_1['1870_id'], truth_confidence_1['1880_id']))
truth_pairs_1[:3]

[('ALB-CN-1870-17478', 'ALB-CN-1880-22724'),
 ('ALB-CN-1870-4121', 'ALB-CN-1880-23484'),
 ('ALB-CN-1870-12513', 'ALB-CN-1880-24928')]

In [220]:
proportion_matched_1 = len(set(matches_pairs) & set(truth_pairs_1)) / len(truth_pairs_1)
proportion_matched_1

0.88

This is a lower proportion matched than confidence 2 and 3, which is to be expected.

##### Confidence Level 0

In [221]:
truth_confidence_0 = truth[truth['confidence'] == 0]
truth_pairs_0 = list(zip(truth_confidence_0['1870_id'], truth_confidence_0['1880_id']))
truth_pairs_0[:3]

[('ALB-CN-1870-16276', 'ALB-CN-1880-20673'),
 ('ALB-CN-1870-10293', 'ALB-CN-1880-31262'),
 ('ALB-CN-1870-21086', 'ALB-CN-1880-5367')]

In [222]:
proportion_matched_0 = len(set(matches_pairs) & set(truth_pairs_0)) / len(truth_pairs_0)
proportion_matched_0

0.4854368932038835

Our proportion matched again goes down at a lower confidence levels, which makes sense because even to an expert these matches are less probable.

In [223]:
confidence_levels = [0, 1, 2, 3,'overall']
match_percentages = [proportion_matched_0, proportion_matched_1, proportion_matched_2, proportion_matched_3, proportion_matched]
threshold_8_df = pd.DataFrame({'confidence_level': confidence_levels, 'match_percentage': match_percentages})
threshold_8_df

,confidence_level,match_percentage
0,0,0.485437
1,1,0.880000
2,2,1.000000
3,3,0.995146
4,overall,0.838527


In [224]:
fig = go.Figure(data=[go.Table(
    header=dict(values=list(threshold_8_df.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[threshold_8_df.confidence_level, round(threshold_8_df.match_percentage, 3)],
               fill_color='lavender',
               align='left'))
])
fig.update_layout(title= "Percent Ground Truth Matched at Match Threshold 0.8")
fig.show()


For the most part, our higher confidence corresponds for higher match percentage after setting  match threshold of 0.8.

### Threshold at 0.90

Repeating all previous steps with an increased threshold.

In [225]:
threshold =  0.90
threshold

0.9

In [226]:
matches = predictions[predictions['match_probability'] > threshold]
matches_pairs = list(zip(matches['unique_id_l'], matches['unique_id_r']))
matches_pairs[:3]

[('ALB-CN-1870-10570', 'ALB-CN-1880-6166'),
 ('ALB-CN-1870-12185', 'ALB-CN-1880-6171'),
 ('ALB-CN-1870-14277', 'ALB-CN-1880-6173')]

In [227]:
proportion_matched = len(set(matches_pairs) & set(truth_pairs)) / len(truth_pairs)
proportion_matched

0.8130311614730878

#### Confidence 3

In [228]:
truth_confidence_3 = truth[truth['confidence'] == 3]
truth_pairs_3 = list(zip(truth_confidence_3['1870_id'], truth_confidence_3['1880_id']))
truth_pairs_3[:3]

[('ALB-CN-1870-1688', 'ALB-CN-1880-22721'),
 ('ALB-CN-1870-1695', 'ALB-CN-1880-22737'),
 ('ALB-CN-1870-1693', 'ALB-CN-1880-22735')]

In [229]:
proportion_matched_3 = len(set(matches_pairs) & set(truth_pairs_3)) / len(truth_pairs_3)
proportion_matched_3

0.9902912621359223

Let's look at what we failed to match a confidence 3

In [230]:
set(truth_pairs_3) - set(matches_pairs)

{('ALB-CN-1870-4694', 'ALB-CN-1880-11783'),
 ('ALB-CN-1870-8855', 'ALB-CN-1880-17581')}

In [231]:
mentions[mentions['mention_id'] == 'ALB-CN-1880-17581'].T

,36986
mention_id,ALB-CN-1880-17581
source,ALB_CN_1880
source_year,1880
county,ALB
original_data,"{""age"": ""18"", ""head"": """", ""line"": ""17581"", ""ra..."
confidence,0.9
full_name,A Susan Timberlake
first_name,A
middle_name,Susan
last_name,Timberlake


In [232]:
mentions[mentions['mention_id'] == 'ALB-CN-1870-8855'].T

,67901
mention_id,ALB-CN-1870-8855
source,ALB_CN_1870
source_year,1870
county,ALB
original_data,"{""age"": ""5"", ""head"": """", ""line"": ""8855"", ""page..."
confidence,0.9
full_name,Susan A Timberlake
first_name,Susan
middle_name,A
last_name,Timberlake


In [233]:
mentions[mentions['mention_id'] == 'ALB-CN-1870-4694'].T

,63958
mention_id,ALB-CN-1870-4694
source,ALB_CN_1870
source_year,1870
county,ALB
original_data,"{""age"": ""23"", ""head"": """", ""line"": ""4694"", ""pag..."
confidence,0.9
full_name,Fannie Abell
first_name,Fannie
middle_name,NaN
last_name,Abell


In [234]:
mentions[mentions['mention_id'] == 'ALB-CN-1880-11783'].T

,5714
mention_id,ALB-CN-1880-11783
source,ALB_CN_1880
source_year,1880
county,ALB
original_data,"{""age"": ""28"", ""head"": """", ""line"": ""11783"", ""ra..."
confidence,0.9
full_name,Fanny W Abell
first_name,Fanny
middle_name,W
last_name,Abell


We again failed to capture the Susan Timberlake due to the first name and middle name mismtach. The other match we failed to catch is Franny[ie] Abell. This could be because in 1880 the norm_first name was encoded as FRANCES but in 1870 it was encoded as 1880, our model have considered this a difference.

#### Confidence 2

In [235]:
truth_confidence_2 = truth[truth['confidence'] == 2]
truth_pairs_2 = list(zip(truth_confidence_2['1870_id'], truth_confidence_2['1880_id']))
truth_pairs_2[:3]

[('ALB-CN-1870-17149', 'ALB-CN-1880-25770'),
 ('ALB-CN-1870-18766', 'ALB-CN-1880-10837'),
 ('ALB-CN-1870-538', 'ALB-CN-1880-16327')]

In [236]:
proportion_matched_2 = len(set(matches_pairs) & set(truth_pairs_2)) / len(truth_pairs_2)
proportion_matched_2

1.0

#### Confidence 1

In [237]:
truth_confidence_1 = truth[truth['confidence'] == 1]
truth_pairs_1 = list(zip(truth_confidence_1['1870_id'], truth_confidence_1['1880_id']))
truth_pairs_1[:3]

[('ALB-CN-1870-17478', 'ALB-CN-1880-22724'),
 ('ALB-CN-1870-4121', 'ALB-CN-1880-23484'),
 ('ALB-CN-1870-12513', 'ALB-CN-1880-24928')]

In [238]:
proportion_matched_1 = len(set(matches_pairs) & set(truth_pairs_1)) / len(truth_pairs_1)
proportion_matched_1

0.84

#### Confidence 3

In [239]:
truth_confidence_0 = truth[truth['confidence'] == 0]
truth_pairs_0 = list(zip(truth_confidence_0['1870_id'], truth_confidence_0['1880_id']))
truth_pairs_0[:3]

[('ALB-CN-1870-16276', 'ALB-CN-1880-20673'),
 ('ALB-CN-1870-10293', 'ALB-CN-1880-31262'),
 ('ALB-CN-1870-21086', 'ALB-CN-1880-5367')]

In [240]:
proportion_matched_0 = len(set(matches_pairs) & set(truth_pairs_0)) / len(truth_pairs_0)
proportion_matched_0

0.4174757281553398

In [241]:
confidence_levels = [0, 1, 2, 3,'overall']
match_percentages = [proportion_matched_0, proportion_matched_1, proportion_matched_2, proportion_matched_3, proportion_matched]
threshold_9_df = pd.DataFrame({'confidence_level': confidence_levels, 'match_percentage': match_percentages})
threshold_9_df

,confidence_level,match_percentage
0,0,0.417476
1,1,0.840000
2,2,1.000000
3,3,0.990291
4,overall,0.813031


In [242]:
fig = go.Figure(data=[go.Table(
    header=dict(values=list(threshold_9_df.columns),
                fill_color='paleturquoise',
                align='left'),
    cells=dict(values=[threshold_9_df.confidence_level, round(threshold_9_df.match_percentage, 3)],
               fill_color='lavender',
               align='left'))
])
fig.update_layout(title= "Percent Ground Truth Matched at Match Threshold 0.9")
fig.show()

For the most part, our higher confidence corresponds for higher match percentage after setting  match threshold of 0.9. Again with the expection of confidence 2 being higher than 3 for match percentage because of the one confidence 3 we failed to match. The match percentages are overall lower, but it makes sense because we were more restrictive on our matches are.